In [ ]:



import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
import os
import math






In [ ]:

BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

#Data loading, transformation, augmentations



transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)





Using device: cuda


100%|██████████| 170M/170M [00:13<00:00, 13.0MB/s]


In [ ]:
def pytorch_conv2d(X, W, b, stride=1, padding=0):


    return F.conv2d(X, W, bias=b, stride=stride, padding=padding)


In [ ]:

# Activation + Loss + Regularization


def relu(Z):
    return Z.clamp(min=0)

def softmax(Z):
    exp_Z = torch.exp(Z - torch.max(Z, dim=0, keepdim=True).values)
    return exp_Z / torch.sum(exp_Z, dim=0, keepdim=True)

def cross_entropy_loss(A, y):
    m = y.shape[0]
    log_probs = -torch.log(A[y, range(m)] + 1e-8)
    return torch.sum(log_probs) / m

def dropout(A, drop_prob, training=True):
    if not training or drop_prob == 0.0:
        return A
    keep_prob = 1.0 - drop_prob
    mask = (torch.rand_like(A) < keep_prob).float()
    return (A * mask) / keep_prob

def l2_regularization_cost(weight_list, lambd):
    l2_cost = 0.0
    for W in weight_list:
        l2_cost += torch.sum(W ** 2)
    return (lambd / 2.0) * l2_cost



In [ ]:

#OPTIMIZERS SGD AND ADAM

class ManualSGDMomentum:
    def __init__(self, params, lr, beta=0.9):
        self.params = params
        self.lr = lr
        self.beta = beta
        self.velocities = [torch.zeros_like(p) for p in self.params]

    def step(self):
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is not None:
                    self.velocities[i] = self.beta * self.velocities[i] + (1 - self.beta) * p.grad
                    p -= self.lr * self.velocities[i]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


class ManualAdam:
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.t = 0
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]

    def step(self):
        self.t += 1
        with torch.no_grad():
            for i, p in enumerate(self.params):
                if p.grad is not None:
                    self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * p.grad
                    self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (p.grad ** 2)
                    m_hat = self.m[i] / (1 - self.beta1 ** self.t)
                    v_hat = self.v[i] / (1 - self.beta2 ** self.t)
                    p -= self.lr * m_hat / (v_hat.sqrt() + self.epsilon)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()



In [ ]:
#LR DECAY

class CosineLRDecay:
    def __init__(self, optimizer, lr_max, lr_min, total_epochs):
        self.optimizer = optimizer
        self.lr_max = lr_max
        self.lr_min = lr_min
        self.total_epochs = total_epochs

    def step(self, epoch):
        new_lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * \
                 (1 + math.cos(math.pi * epoch / self.total_epochs))
        self.optimizer.lr = new_lr
        return new_lr


class ExponentialLRDecay:
    def __init__(self, optimizer, lr_init, gamma=0.95):
        self.optimizer = optimizer
        self.lr_init = lr_init
        self.gamma = gamma

    def step(self, epoch):
        new_lr = self.lr_init * (self.gamma ** epoch)
        self.optimizer.lr = new_lr
        return new_lr





In [ ]:

def _make_conv_params(c_in, c_out):
    """Helper: create weight, bias, and BN params for one conv layer."""



    w = torch.nn.Parameter(
        torch.randn(c_out, c_in, 3, 3, device=DEVICE) * math.sqrt(2.0 / (c_in * 3 * 3)))
    b = torch.nn.Parameter(torch.zeros(c_out, device=DEVICE))
    bn_gamma = torch.nn.Parameter(torch.ones(c_out, device=DEVICE))
    bn_beta  = torch.nn.Parameter(torch.zeros(c_out, device=DEVICE))
    bn_rm    = torch.zeros(c_out, device=DEVICE)
    bn_rv    = torch.ones(c_out, device=DEVICE)
    return w, b, bn_gamma, bn_beta, bn_rm, bn_rv



In [ ]:
"""
Using pytorch libraries and tools for Architecture
VGG-style CNN v3 nn.Module version
=====================================
:

  Block 1: Conv(3→64) → BN → ReLU → Conv(64→64) → BN → ReLU → MaxPool → Dropout(0.1)

  Block 2: Conv(64→128) → BN → ReLU → Conv(128→128) → BN → ReLU → MaxPool → Dropout(0.2)

  Block 3: Conv(128→256) → BN → ReLU → Conv(256→256) → BN → ReLU → MaxPool → Dropout(0.3)

  Block 4: Conv(256→512) → BN → ReLU → Conv(512→512) → BN → ReLU → MaxPool → Dropout(0.4)

  GAP → FC(512→256) → ReLU → Dropout(0.5) → FC(256→10)



"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class VGGStyleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # ── Block 1: 3 → 64 → 64, pool ──
        self.conv1a = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(64)

        # ── Block 2: 64 → 128 → 128, pool ──
        self.conv2a = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(128)

        # ── Block 3: 128 → 256 → 256, pool ──
        self.conv3a = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3a = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3b = nn.BatchNorm2d(256)

        # ── Block 4: 256 → 512 → 512, pool ──
        self.conv4a = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4a = nn.BatchNorm2d(512)
        self.conv4b = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn4b = nn.BatchNorm2d(512)

        # ── Classifier ──
        self.fc1 = nn.Linear(512, 256)
        self.fc2 = nn.Linear(256, num_classes)

        # Initialize weights (He initialization, same as your original)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)   # gamma = 1
                nn.init.zeros_(m.bias)    # beta = 0
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                nn.init.zeros_(m.bias)




    def forward(self, x):



        # Block 1: (N,3,32,32) → (N,64,16,16)
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = F.relu(self.bn1b(self.conv1b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.1, training=self.training)




        # Block 2: (N,64,16,16) → (N,128,8,8)
        x = F.relu(self.bn2a(self.conv2a(x)))
        x = F.relu(self.bn2b(self.conv2b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.2, training=self.training)




        # Block 3: (N,128,8,8) → (N,256,4,4)
        x = F.relu(self.bn3a(self.conv3a(x)))
        x = F.relu(self.bn3b(self.conv3b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.3, training=self.training)




        # Block 4: (N,256,4,4) → (N,512,2,2)
        x = F.relu(self.bn4a(self.conv4a(x)))
        x = F.relu(self.bn4b(self.conv4b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.4, training=self.training)





        # GAP: (N,512,2,2) → (N,512)
        x = F.adaptive_avg_pool2d(x, 1)
        x = x.view(x.size(0), -1)




        # Classifier
        x = F.relu(self.fc1(x))
        x = F.dropout(x, 0.5, training=self.training)
        x = self.fc2(x)          # raw logits — use nn.CrossEntropyLoss

        return x


In [ ]:
#checking the paramerters of the model


if __name__ == "__main__":

    model = VGGStyleCNN()

    print(model)

    print()




    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")





    # quick forward
    dummy = torch.randn(2, 3, 32, 32)
    out = model(dummy)
    print(f"Output shape: {out.shape}")   # expect (2, 10)

VGGStyleCNN(
  (conv1a): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1a): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv1b): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1b): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2a): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2a): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2b): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2b): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3a): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3a): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3b): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3b): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=T

In [ ]:

# Evaluation


def evaluate(model, loader):
    correct = 0
    total = 0


    with torch.no_grad():

        for inputs, labels in loader:

            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            logits = model.forward(inputs, training=False, drop_prob=0.0)
            predictions = torch.argmax(logits, dim=0)
            correct += (predictions == labels).sum().item()
            total += labels.shape[0]

    return 100.0 * correct / total



In [ ]:

"""
Training for VGG-style CNN ,nn.Module version
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
import os
import math


#Setup
BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

#Transform
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])
#datasets
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)


# ─────────────────────────────────────────────
# Evaluation — [CHANGED] slightly simplified
# ─────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()

    correct = 0
    total = 0



    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            logits = model(inputs)

            predictions = logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)



    #change if to train mode
    model.train()
    return 100.0 * correct / total



Using device: cuda


In [ ]:

# ─────────────────────────────────────────────
# Training function
# ─────────────────────────────────────────────
def train_model(epochs=20,
                lr=0.002,
                weight_decay=0.0005,    # [CHANGED] this replaces your manual L2 regularization
                cosine_lr_min=1e-4):

    print("=" * 65)
    print(f"LR: {lr} | Weight Decay: {weight_decay}")
    print(f"Cosine schedule: {lr} → {cosine_lr_min}")
    print(f"Epochs: {epochs}")
    print("=" * 65)

    # ── Model ──
    model = VGGStyleCNN().to(DEVICE)

    # ── Loss ──
    criterion = nn.CrossEntropyLoss()  # pytorch cross entropy loss from mymanual softmax + cross_entropy_loss
                                        # it is a  log-softmax and negative log likelihood internally








    # ── Optimizer ──
    # PyTorch's Adam is better than my ManualAdam
    #
    # weight_decay parameter so l2_regularization_cost() is useless now
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)







    # ── LR Scheduler ──
    # changed to PyTorch's CosineAnnealingLR  from my CosineLRDecay

    # lr to cosine_lr_min over total epochs




    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=cosine_lr_min
    )








    cost_values = []
    train_acc_history = []
    test_acc_history = []
    lr_history = []
    best_test_acc = 0.0
    total_batches = len(train_loader)







    for epoch in range(epochs):
        epoch_start = time.time()
        model.train()

        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            # Forward
            logits = model(inputs)              # (N, 10) — note: no softmax, no transpose
            loss = criterion(logits, labels)    #this  replaces softmax + CE + L2

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            cost_values.append(loss.item())

            if (batch_idx + 1) % 50 == 0 or (batch_idx + 1) == total_batches:
                print(f"  Epoch {epoch+1}/{epochs} | "
                      f"Batch {batch_idx+1}/{total_batches} | "
                      f"Loss: {loss.item():.4f}", end='\r')





        # PyTorch scheduler can track the epcoh
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]
        lr_history.append(current_lr)







        # Evaluate
        print(f"\n  Evaluating epoch {epoch+1}...", end='\r')
        train_acc = evaluate(model, train_loader)
        test_acc = evaluate(model, test_loader)



        train_acc_history.append(train_acc)
        test_acc_history.append(test_acc)
        best_test_acc = max(best_test_acc, test_acc)



        elapsed = time.time() - epoch_start
        print(f"  Epoch {epoch+1:>3d}/{epochs} | "
              f"Train Acc: {train_acc:6.2f}% | "
              f"Test Acc: {test_acc:6.2f}% | "
              f"Best: {best_test_acc:6.2f}% | "
              f"Loss: {cost_values[-1]:.4f} | "
              f"LR: {current_lr:.6f} | "
              f"Time: {elapsed:.1f}s")






    #  Save the trained model!!!!
    torch.save(model.state_dict(), "vgg_cifar10_trained.pth")
    print("Model saved to vgg_cifar10_trained.pth")

    return model, cost_values, train_acc_history, test_acc_history, lr_history



In [ ]:

# Run




if __name__ == "__main__":
    epochs = 20

    start_time = time.time()
    model, cost_values, train_acc, test_acc, lr_hist = train_model(
        epochs=epochs,
        lr=0.002,
        weight_decay=0.0005,
        cosine_lr_min=1e-4,
    )
    total_time = time.time() - start_time





    print(f"\nTotal training time: {total_time:.2f} seconds")

    print(f"Final Train Accuracy: {train_acc[-1]:.2f}%")

    print(f"Final Test  Accuracy: {test_acc[-1]:.2f}%")

    print(f"Best  Test  Accuracy: {max(test_acc):.2f}%")







    #    Plots
    os.makedirs("plots", exist_ok=True)











    plt.figure()
    plt.plot(range(1, epochs + 1), train_acc, marker='o', markersize=2)
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Training Accuracy')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/train_accuracy.png", dpi=150)
    plt.close()








    plt.figure()
    plt.plot(range(1, epochs + 1), test_acc, marker='s', markersize=2, color='orange')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title('Test Accuracy')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/test_accuracy.png", dpi=150)
    plt.close()








    plt.figure()
    plt.plot(range(len(cost_values)), cost_values, linewidth=0.3)
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/training_loss.png", dpi=150)
    plt.close()






    plt.figure()
    plt.plot(range(1, epochs + 1), lr_hist, marker='d', markersize=2, color='green')
    plt.xlabel('Epoch')
    plt.ylabel('Learning Rate')
    plt.title('Learning Rate Schedule (Cosine Decay)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/lr_schedule.png", dpi=150)
    plt.close()

    print("Saved plots to plots/ directory.")

LR: 0.002 | Weight Decay: 0.0005
Cosine schedule: 0.002 → 0.0001
Epochs: 20
  Epoch 1/20 | Batch 391/391 | Loss: 1.5346
  Epoch   1/20 | Train Acc:  48.93% | Test Acc:  48.73% | Best:  48.73% | Loss: 1.5346 | LR: 0.001988 | Time: 36.2s
  Epoch 2/20 | Batch 391/391 | Loss: 0.9646
  Epoch   2/20 | Train Acc:  56.53% | Test Acc:  56.77% | Best:  56.77% | Loss: 0.9646 | LR: 0.001954 | Time: 34.6s
  Epoch 3/20 | Batch 391/391 | Loss: 0.9585
  Epoch   3/20 | Train Acc:  54.68% | Test Acc:  54.42% | Best:  56.77% | Loss: 0.9585 | LR: 0.001896 | Time: 34.4s
  Epoch 4/20 | Batch 391/391 | Loss: 1.0359
  Epoch   4/20 | Train Acc:  72.08% | Test Acc:  71.84% | Best:  71.84% | Loss: 1.0359 | LR: 0.001819 | Time: 34.6s
  Epoch 5/20 | Batch 391/391 | Loss: 0.8777
  Epoch   5/20 | Train Acc:  68.09% | Test Acc:  67.93% | Best:  71.84% | Loss: 0.8777 | LR: 0.001722 | Time: 34.6s
  Epoch 6/20 | Batch 391/391 | Loss: 0.8596
  Epoch   6/20 | Train Acc:  75.58% | Test Acc:  74.18% | Best:  74.18% | Loss: 